# Net worth over time

Reads the latest run file for the current scenario under `data/shared/state/runs/` and plots month-end net worth (nominal) plus an inflation-adjusted (CPI) series.

The scenario is resolved the same way the `sim` CLI resolves it: `$ANALYSIS_SCENARIO` if set, else the name recorded in the data repo's `.current_scenario` (written by `sim load <scenario>`), else `current`.

The run file is opened read-only via `RunConnection`; all reads go through `rc.items(...)`, which returns tidy DataFrames suitable for pandas groupby / pivot. See [ADR-002](../docs/architecture/ADR-002-storage-schema.md) for the underlying schema.

> **Note:** after running cells, `git status` will show this notebook as modified even with no real changes. Staging it (`git add`) will make it show unmodified again, unless actual changes are present — see [notebooks/README.md](./README.md#git-status-shows-a-notebook-as-modified-after-running-it) for why.


In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown

from analysis.sim.connection import RunConnection
from analysis.sim.loader import resolve_scenario_name

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Same resolution the CLI uses: $ANALYSIS_SCENARIO, else the data repo's
# .current_scenario, else "current".
SCENARIO_REF = resolve_scenario_name(
    os.environ.get("ANALYSIS_DATA_DIR", REPO / "data" / "shared" / "sim"),
    os.environ.get("ANALYSIS_SCENARIO") or None,
)
display(SCENARIO_REF)

DATA_DIR = SCENARIO_REF.data_dir
SCENARIO = SCENARIO_REF.name
RUNS_DIR = SCENARIO_REF.runs_dir
if SCENARIO_REF.run_path is None:
    raise FileNotFoundError(f"no runs for {SCENARIO!r} in {RUNS_DIR}; run `sim run`")
RUN_FILE = SCENARIO_REF.run_path

rc = RunConnection(RUN_FILE)

plot_mode = widgets.ToggleButtons(
    options=[("Nominal", "nominal"), ("CPI-constant", "cpi_constant")],
    value="cpi_constant",
    description="Dollar basis:",
)
# display(plot_mode)
expenses_caveat = widgets.Output()
with expenses_caveat:
    display(
        Markdown(
            "_Note that expenses in CPI-constant dollars decrease slightly, because constant-valued expenses scale down with inflation._  \n"
            "_For example, fixed-rate mortgage expenses do not increase with inflation, so after compensating for inflation, they appear smaller._  \n"
            "_Fixed expenses are a small part of the total, so the overall effect is small, but it is present._  \n"
        )
    )


In [ ]:
_META_COLS = [
    "scenario_name", "status", "started_at", "finished_at",
    "code_version", "code_dirty", "data_version", "data_dirty",
    "data_mode", "run_by",
]
run_meta = rc.items("simulation_run")[_META_COLS].copy()
run_meta["code_version"] = run_meta["code_version"].str[:12]
run_meta["data_version"] = run_meta["data_version"].str[:12]
run_meta.T


In [ ]:
nw = rc.items("net_worth").rename(columns={"net_worth": "nominal_cents"})
cpi = rc.items("net_worth_constant_cpi")[
    ["month", "constant_cents", "cumulative_index_micros"]
]
nw = nw.merge(cpi, on="month", how="left")

nw["month"] = pd.to_datetime(nw["month"])
nw["nominal"] = nw["nominal_cents"] / 100
nw["cpi_constant"] = nw["constant_cents"].astype("Float64") / 100
nw = nw.set_index("month")
_cpi_index = nw["cumulative_index_micros"]


def _cpi_constant(monthly):
    """Convert a monthly dollar series to the run's CPI-constant dollars."""
    factor = 1_000_000 / _cpi_index.reindex(monthly.index)
    return monthly.mul(factor, axis=0)


def _annual_cpi_constant(yearly):
    year_end_index = _cpi_index.groupby(_cpi_index.index.year).last()
    return yearly.mul(1_000_000 / year_end_index.reindex(yearly.index), axis=0)

nw[["nominal", "cpi_constant"]].head()


## Year-end net worth (table)


In [ ]:
year_end = (
    nw.groupby(nw.index.year)
      .tail(1)
      .assign(year=lambda d: d.index.year)
      .set_index("year")[["nominal", "cpi_constant"]]
)
year_end.style.format({"nominal": "${:,.0f}", "cpi_constant": "${:,.0f}"})


## Monthly net worth (chart)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(nw.index, nw["nominal"], label="nominal")
if nw["cpi_constant"].notna().any():
    ax.plot(nw.index, nw["cpi_constant"], label="CPI-constant", linestyle="--")
ax.set_ylim(bottom=0)
ax.set_title(
    f"Net worth — {run_meta.loc[0, 'scenario_name']} "
    f"({RUN_FILE.stem})"
)
ax.set_xlabel("month")
ax.set_ylabel("USD")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
ax.grid(True, axis="y", alpha=0.3)
ax.legend()
plt.xticks(rotation=90)
plt.show()

## Income by category

Income flows are transfers _out of_ accounts with `kind = 'income'`. Subkinds are bucketed:

- **salary** — `salary`
- **investment** — `interest`, `dividend`, `capital_gain`
- **other** — `ss_benefit`, `pension`, `rental`, `other_income`


In [ ]:
_INCOME_SUBKIND_TO_CAT = {
    "salary": "salary",
    "interest": "investment",
    "dividend": "investment",
    "capital_gain": "investment",
    "ss_benefit": "social security",
}
_INCOME_CATS = ["salary", "social security", "investment"]

xfer = rc.items("transfer_event", from_kind="income").copy()
xfer["month"] = pd.to_datetime(xfer["month"])
xfer["category"] = xfer["from_subkind"].map(_INCOME_SUBKIND_TO_CAT).fillna("other")

income = (
    xfer.groupby(["month", "category"])["amount"].sum()
        .unstack("category", fill_value=0)
        .reindex(columns=_INCOME_CATS, fill_value=0)
    / 100
)
income["total"] = income.sum(axis=1)
income.head()


In [ ]:
income_yearly = income.groupby(income.index.year).sum()
income_yearly.index.name = "year"

# Partial-first-year catch-up (salary/investment/SS already received before
# scenario.start) is authored as scenario data now, not back-calculated here:
# `ytd_amount`/`ytd_preroll` on scheduled.csv rows, or year_to_date.csv for
# everything else. The engine posts those as ordinary transfer_event rows
# (ADR-003 §4.11), so they're already included in `income` above.

income_yearly["total"] = income_yearly[_INCOME_CATS].sum(axis=1)
income_yearly.style.format("${:,.0f}")


## Income by type

Charts use the `PLOT_MODE` selected in the setup cell (`nominal` or `cpi_constant`).


In [ ]:
categories = ["salary", "social security", "investment"]


def _plot_income(mode):
    mode_label = "CPI-constant" if mode == "cpi_constant" else "nominal"
    income_chart = (
        _cpi_constant(income).groupby(lambda month: month.year).sum()
        if mode == "cpi_constant" else income_yearly
    )
    fig, ax = plt.subplots(figsize=(10, 5))
    income_chart[categories].plot(
        kind="bar", stacked=True, ax=ax,
        color=["#1f77b4", "#2ca02c", "#ff7f0e"],
    )
    ax.set_title(
        f"Annual income by category ({mode_label}) — {run_meta.loc[0, 'scenario_name']} "
        f"({RUN_FILE.stem})"
    )
    ax.set_xlabel("year")
    ax.set_ylabel("USD")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(title="category")
    plt.show()


income_plot = widgets.interactive_output(_plot_income, {"mode": plot_mode})
display(widgets.VBox([plot_mode, income_plot]))


## Expenses by category

Expense flows are transfers _into_ accounts with `kind = 'expense'`. Subkinds are bucketed:

- **housing** — `housing`, `interest_expense`
- **healthcare** — `healthcare`
- **taxes** — `taxes_federal`, `taxes_state`, `taxes_property`
- **living** — `living`
- **other** — `other_expense`

The built-in `depreciation` account is excluded: it's a book entry recording value lost by wasting assets (vehicles), not money spent, so counting it here would inflate expenses and understate the after-tax surplus.


In [ ]:
_EXPENSE_SUBKIND_TO_CAT = {
    "housing": "housing",
    "interest_expense": "housing",
    "taxes_property": "housing",
    "healthcare": "healthcare",
    "taxes_federal": "taxes",
    "taxes_state": "taxes",
    "living": "living",
    "food": "living",
}
_EXPENSE_CATS = ["housing", "healthcare", "taxes", "living", "other"]
# Depreciation is a book entry, not spending; it would otherwise inflate every
# cash-expense total and understate the after-tax surplus below.
NON_CASH_EXPENSE_ACCOUNTS = {"depreciation"}

xfer_exp = rc.items("transfer_event", to_kind="expense").copy()
xfer_exp = xfer_exp[~xfer_exp["to_name"].isin(NON_CASH_EXPENSE_ACCOUNTS)]
xfer_exp["month"] = pd.to_datetime(xfer_exp["month"])
xfer_exp["category"] = (
    xfer_exp["to_subkind"].map(_EXPENSE_SUBKIND_TO_CAT).fillna("other")
)

expenses = (
    xfer_exp.groupby(["month", "category"])["amount"].sum()
        .unstack("category", fill_value=0)
        .reindex(columns=_EXPENSE_CATS, fill_value=0)
    / 100
)
expenses["total"] = expenses.sum(axis=1)
expenses_yearly = expenses.groupby(expenses.index.year).sum()
expenses_yearly.index.name = "year"
expenses_yearly.style.format("${:,.0f}")


In [ ]:

def _plot_expenses(mode):
    mode_label = "CPI-constant" if mode == "cpi_constant" else "nominal"
    expenses_chart = (
        _cpi_constant(expenses).groupby(lambda month: month.year).sum()
        if mode == "cpi_constant" else expenses_yearly
    )
    fig, ax = plt.subplots(figsize=(10, 5))
    expenses_chart[_EXPENSE_CATS].plot(
        kind="bar", stacked=True, ax=ax,
        color=["#1f77b4", "#d62728", "#9467bd", "#8c564b", "#7f7f7f"],
    )
    ax.set_title(
        f"Annual expenses by category ({mode_label}) — {run_meta.loc[0, 'scenario_name']} "
        f"({RUN_FILE.stem})"
    )
    ax.set_xlabel("year")
    ax.set_ylabel("USD")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(title="category")
    plt.show()


expenses_plot = widgets.interactive_output(_plot_expenses, {"mode": plot_mode})
display(widgets.VBox([plot_mode, expenses_plot, expenses_caveat]))


## Realized vs. unrealized gains

Splits every income-side flow into three buckets, to check whether actual cash distributions (not paper growth) cover expenses:

- **taxable** — reaches a liquid account (`checking`/`savings`) with a taxable character (`ordinary`, `ltcg`, `tax_deferred_out`), plus the *taxable portion* of Social Security.
- **non-taxable** — reaches a liquid account with a `tax_free` character (Roth growth/withdrawals, tax-free income), plus the *non-taxable portion* of Social Security.
- **unrealized** — income-kind growth (interest, dividends, appreciation) that stays inside a retirement/brokerage account and never reaches cash.

Social Security is **not** simply lumped into "taxable": only up to 85% of it ever is (per the federal provisional-income test — see `_taxable_social_security_cents`), and it can be 0% at low enough income. We split each year's SS benefit using the engine's own computed `federal_taxable_ss` (from `tax_year_summary`), not a flat assumption.

Social Security is also counted **gross**: Medicare Part B/D premiums are withheld before the deposit, so those legs go straight from the SS income account to a healthcare expense account and never touch cash. They still count as received (and taxable) income, so `ss_benefit` is bucketed on tax character alone rather than on reaching a liquid account.

Caveats (this is a first pass — flag anything that doesn't match your mental model):

- `return_of_capital` (basis returned on a sale) and `tax_deferred_in` (contributions) are excluded entirely: they're principal movements, not gains.
- "Liquid" = `checking`/`savings` subkinds only; a distribution landing in, say, an HSA cash account wouldn't count as realized here.
- The scenario's final year has no computed `federal_taxable_ss` yet (reconciliation happens the following April), so that year's SS is left entirely in "non-taxable" — an understatement, consistent with the same caveat in "After-tax income" below.
- `distribution_coverage` below is `taxable + non-taxable − expenses`; negative years are ones where realized cash distributions alone didn't cover spending (the gap was plugged by `waterfall_fill`/`refill_transfer` principal draws, which aren't gains and so aren't counted as coverage here). Medicare premiums withheld from Social Security are in both sides of that subtraction, as they should be.


In [ ]:
LIQUID_SUBKINDS = {"checking", "savings"}
TAXABLE_GAIN_CHARS = {"ordinary", "ltcg", "tax_deferred_out"}  # ss_benefit handled below
NONTAXABLE_GAIN_CHARS = {"tax_free"}

gains = rc.items("transfer_event")
gains = gains[gains["reason_kind"] != "initial"].copy()
gains["month"] = pd.to_datetime(gains["month"])
gains["year"] = gains["month"].dt.year

reaches_cash = gains["to_subkind"].isin(LIQUID_SUBKINDS)
is_unrealized = (gains["from_kind"] == "income") & ~reaches_cash
is_taxable = reaches_cash & gains["tax_character"].isin(TAXABLE_GAIN_CHARS)
is_nontaxable = reaches_cash & gains["tax_character"].isin(NONTAXABLE_GAIN_CHARS)
# Gross benefit, including the Medicare premiums withheld before deposit --
# those legs never reach cash but are still received (and taxed) income.
is_ss = gains["tax_character"] == "ss_benefit"

gains["gain_category"] = pd.NA
gains.loc[is_unrealized, "gain_category"] = "unrealized"
gains.loc[is_taxable, "gain_category"] = "taxable"
gains.loc[is_nontaxable, "gain_category"] = "non-taxable"
gains.loc[is_ss, "gain_category"] = "social_security"
gains = gains.dropna(subset=["gain_category"])

_GAIN_CATS = ["taxable", "non-taxable", "unrealized"]
gains_yearly = (
    gains.groupby(["year", "gain_category"])["amount"].sum()
        .unstack("gain_category", fill_value=0)
        .reindex(columns=[*_GAIN_CATS, "social_security"], fill_value=0)
    / 100
)
gains_yearly.index.name = "year"

# Split SS by the engine's own provisional-income test, not a flat assumption
# (see markdown above): only federal_taxable_ss of it is actually taxable.
_tax_summary_for_ss = rc.items("tax_year_summary")
federal_taxable_ss = (
    _tax_summary_for_ss.loc[
        _tax_summary_for_ss["bucket"] == "federal_taxable_ss", ["year", "amount"]
    ].set_index("year")["amount"]
    / 100
)
ss_total = gains_yearly["social_security"]
ss_taxable = federal_taxable_ss.reindex(gains_yearly.index).fillna(0).clip(upper=ss_total)
ss_nontaxable = ss_total - ss_taxable

gains_yearly["taxable"] += ss_taxable
gains_yearly["non-taxable"] += ss_nontaxable
gains_yearly = gains_yearly.drop(columns="social_security")

gains_yearly["realized_total"] = gains_yearly["taxable"] + gains_yearly["non-taxable"]
gains_yearly["expenses"] = expenses_yearly["total"].reindex(gains_yearly.index).fillna(0)
gains_yearly["distribution_coverage"] = gains_yearly["realized_total"] - gains_yearly["expenses"]
gains_yearly.style.format("${:,.0f}")


In [ ]:
def _plot_gains(mode):
    mode_label = "CPI-constant" if mode == "cpi_constant" else "nominal"
    gains_chart = (
        _annual_cpi_constant(gains_yearly)
        if mode == "cpi_constant" else gains_yearly
    )
    fig, ax = plt.subplots(figsize=(10, 5))
    years = gains_chart.index.astype(str)
    gains_chart[_GAIN_CATS].plot(
        kind="bar", stacked=True, ax=ax,
        color=["#2ca02c", "#1f77b4", "#7f7f7f"],
    )
    ax.plot(
        years, gains_chart["expenses"],
        label="expenses", color="#d62728", marker="o", linewidth=2,
    )
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.set_title(
        f"Realized vs. unrealized gains ({mode_label}) — {run_meta.loc[0, 'scenario_name']} "
        f"({RUN_FILE.stem})"
    )
    ax.set_xlabel("year")
    ax.set_ylabel("USD")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(title="category")
    plt.setp(ax.get_xticklabels(), rotation=90)
    plt.show()


gains_plot = widgets.interactive_output(_plot_gains, {"mode": plot_mode})
display(widgets.VBox([plot_mode, gains_plot, expenses_caveat]))


## After-tax income

After-tax income is gross income less the year's computed federal and state tax liabilities. `tax_free` priors are excluded: they are tax inputs, not a substitute for after-tax income.

> **Tax timing:** tax liability is attributed to the tax year in which income occurs. Withholding and estimated payments are cash movements in that same year; the annual reconciliation payment or refund occurs in April of the following year. Do not interpret the reconciliation year as the tax year.

The engine runs a hidden tax-only tail after the declared scenario end, so the final scenario year has a computed tax liability without displaying a later simulation year.

The final column compares after-tax income with non-tax expenses, so estimated tax liabilities are not counted twice. The chart uses the `PLOT_MODE` selected in the setup cell.


In [ ]:
tax_summary = rc.items("tax_year_summary")
tax_liability = (
    tax_summary.loc[
        tax_summary["bucket"].isin(["federal_tax_liability", "state_tax_liability"]),
        ["year", "amount"],
    ]
    .groupby("year")["amount"]
    .sum()
    / 100
)

tax_reconciliation = (
    tax_summary.loc[
        tax_summary["bucket"].isin([
            "federal_tax_liability", "state_tax_liability",
            "federal_tax_paid", "state_tax_paid",
            "federal_tax_due", "state_tax_due",
            "federal_tax_overpaid", "state_tax_overpaid",
        ]),
        ["year", "bucket", "amount"],
    ]
    .pivot_table(index="year", columns="bucket", values="amount", aggfunc="sum", fill_value=0)
    .rename(columns={
        "federal_tax_liability": "federal_liability",
        "state_tax_liability": "state_liability",
        "federal_tax_paid": "federal_withheld_or_paid",
        "state_tax_paid": "state_withheld_or_paid",
        "federal_tax_due": "federal_reconciliation_due",
        "state_tax_due": "state_reconciliation_due",
        "federal_tax_overpaid": "federal_estimated_refund",
        "state_tax_overpaid": "state_estimated_refund",
    })
    / 100
)
tax_reconciliation["reconciliation_year"] = tax_reconciliation.index + 1
money_columns = [
    column for column in tax_reconciliation.columns
    if column != "reconciliation_year"
]
display(tax_reconciliation.style.format({column: "${:,.0f}" for column in money_columns}))

net = pd.DataFrame({
    "gross_income": income_yearly["total"],
    "tax_liability": tax_liability,
    "expenses": expenses_yearly["total"],
    "tax_expenses": expenses_yearly["taxes"],
})
net[["gross_income", "expenses", "tax_expenses"]] = net[[
    "gross_income", "expenses", "tax_expenses",
]].fillna(0)
net["after_tax_income"] = net["gross_income"] - net["tax_liability"]
net["non_tax_expenses"] = net["expenses"] - net["tax_expenses"]
net["after_tax_surplus"] = net["after_tax_income"] - net["non_tax_expenses"]
net.style.format("${:,.0f}")

In [ ]:
def _plot_net(mode):
    mode_label = "CPI-constant" if mode == "cpi_constant" else "nominal"
    net_chart = (
        _annual_cpi_constant(net)
        if mode == "cpi_constant" else net
    )
    fig, ax = plt.subplots(figsize=(10, 5))
    years = net_chart.index.astype(str)
    ax.bar(
        years, net_chart["after_tax_income"],
        label="after-tax income", color="#2ca02c", alpha=0.7,
     )
    ax.bar(
        years, -net_chart["non_tax_expenses"],
        label="non-tax expenses", color="#d62728", alpha=0.7,
     )
    ax.plot(
        years, net_chart["tax_liability"],
        label="tax liability", color="#9467bd", marker="o", linewidth=2,
     )
    ax.plot(
        years, net_chart["after_tax_surplus"],
        label="after-tax surplus", color="black", marker="o", linewidth=2,
     )
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.set_title(
        f"Annual after-tax income ({mode_label}) — {run_meta.loc[0, 'scenario_name']} "
        f"({RUN_FILE.stem})"
    )
    ax.set_xlabel("year")
    ax.set_ylabel("USD")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"${v:,.0f}"))
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend()
    plt.setp(ax.get_xticklabels(), rotation=90)
    plt.show()


net_plot = widgets.interactive_output(_plot_net, {"mode": plot_mode})
display(widgets.VBox([plot_mode, net_plot, expenses_caveat]))
